# 2 - Preprocessing, Experiment 5

Experiment 5 (`EX_11`) is Experiment 1b's binaural localization task with the left/right judgement
moved onto the **UP / DOWN** keys. Each participant ran it **twice**, once under each key mapping
(`upL`: UP = left; `upR`: UP = right), each time on a different half of the image pool. Two
recordings, two tracker clocks, one participant.

Everything that did not have to change is imported from `src/preprocessing.py` and
`src/preprocessing_alt.py`, as in `01_preprocessing.ipynb`. What is new lives in
`src/preprocessing_ex11.py`: session pairing, key decoding, structural training removal, and
numbering that runs across a participant's two recordings. Its module docstring explains each.

| | Exp 1b (`EX_9`) | Exp 5 (`EX_11`) |
|---|---|---|
| response keys | LEFT / RIGHT | **UP / DOWN**, meaning fixed per session |
| recordings per participant | one | **two**, `upL` and `upR`, counterbalanced |
| images per session | all 200 | **half**, disjoint between the two sessions |
| blocks | — | 5 per session, 10 over the visit |
| what the marker says | the answer | the **key**; the side needs `RESPONSE_MAPPING` |

**Inputs**  `data_raw/EX_11/asc/*.asc` and, for the checks, `data_raw/EX_11/csv/beh_*.csv`
**Output**  `data_preprocessed/EX_11/long_format_ex11_all.csv`

**The check that must pass before any Exp 5 number is read** is the decoding check below. A
mapping applied the wrong way round inverts every answer in a session, and would look like a
large, clean effect rather than like an error.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, 'src')

import config
import exclusions
import preprocessing_ex11 as pe11

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 80)

## Configuration

The preprocessing config lives in `src/preprocessing_ex11.py` (`EX11_PREPROCESSING_CONFIG`),
because the module and its validation helpers both read it. This cell only selects and reports.

`EXCLUDE_SUBJECTS` here drops only participants **without a usable pair of sessions** — they cannot
be assembled into one table at all. Exclusions made for data-quality reasons are applied later, in
the analysis notebooks, through `src/exclusions.py`.

In [2]:
# ====================================================================
# CONFIGURATION
# ====================================================================

cfg = pe11.EX11_PREPROCESSING_CONFIG

# Participants without a usable pair of sessions:
#   E11016 -- both recordings are labelled half 1, and the behavioural .csv confirm the SAME
#             100 images were shown twice (every other participant: 0 shared). The second session
#             is a repeat viewing, not a mislabelled half 2, so the two cannot be paired.
#   E11026 -- only one (upR) recording exists, and it never reached the experiment stage
#             (training only; both behavioural .csv hold 0 experiment trials).
#   E11037 -- only one (upL) session was recorded; the upR session does not exist.
#   E11053 -- both sessions ran under upR (the upL attempt between them is a false start with
#             0 trials), so the participant has no upL session.
EXCLUDE_SUBJECTS = ('E11016', 'E11026', 'E11037', 'E11053')
EYE = 'R'

print(f"{cfg['name']}: {cfg['description']}\n")
print(f"  sound markers    : {cfg['sound_markers']}")
print(f"  answer markers   : {cfg['answer_markers']}  (a KEY, not a side)")
print(f"  key mappings     : {cfg['response_mappings']}   UP means {pe11.UP_MEANS}")
print(f"  training removal : {cfg['training_removal']}")
print(f"  expected trials  : {cfg['trials_per_session']} per session, {cfg['trials_per_block']} per block")
print(f"  raw              : {cfg['asc_directory']}")
print(f"  output           : data_preprocessed/{cfg['name']}/{cfg['long_format_file']}")

os.makedirs(f"data_preprocessed/{cfg['name']}", exist_ok=True)

EX_11: Sound localization (left/right) answered with UP/DOWN, two counterbalanced key mappings per subject

  sound markers    : ['PLAYED_SOUND_LEFT', 'PLAYED_SOUND_RIGHT', 'PLAYED_SOUND_EMPTY']
  answer markers   : ['ANSWER_UP', 'ANSWER_DOWN']  (a KEY, not a side)
  key mappings     : ('upL', 'upR')   UP means {'upL': 'left', 'upR': 'right'}
  training removal : last_stage
  expected trials  : 100 per session, 20 per block
  raw              : data_raw/EX_11/asc
  output           : data_preprocessed/EX_11/long_format_ex11_all.csv


## 0. EDF → ASC (only if `.EDF` files are present)

The deposit ships the converted `.asc`, so this step normally does nothing. If the original `.EDF`
are in `data_raw/EX_11/edf/`, it runs SR Research's `edf2asc` over any recording without an
`.asc`, with **default flags** (events *and* samples) — the settings every `.asc` in this study was
produced with. Already-converted files are skipped.

In [3]:
# ====================================================================
# EDF -> ASC
# ====================================================================

if os.path.isdir(cfg['edf_directory']):
    written = pe11.convert_edf_files(cfg=cfg)
    print(f"{len(written)} new .asc written")
else:
    print(f"No {cfg['edf_directory']} - using the .asc as deposited.")
n_asc = len([f for f in os.listdir(cfg['asc_directory']) if f.endswith('.asc')])
print(f"{n_asc} .asc in {cfg['asc_directory']}")

No data_raw/EX_11/edf - using the .asc as deposited.
117 .asc in data_raw/EX_11/asc


## Sessions

A participant is a **pair** of recordings. They are paired by the participant ID in the filename
(`E11001_upL_half1`) and ordered by `session_half`, which the dialog fixes before the recording
starts. Because ordering by the half is an assumption about how the sessions were run,
`discover_sessions` **checks it** against the behavioural `.csv` filename timestamps and raises if
any participant's half 2 was recorded before their half 1.

In [4]:
# ====================================================================
# DISCOVER AND PAIR THE SESSIONS
# ====================================================================

sessions = pe11.discover_sessions(exclude_subjects=EXCLUDE_SUBJECTS, cfg=cfg)
session_table = pe11.describe_sessions(sessions, cfg=cfg)
display(session_table)

print(f"{len(sessions)} participants, {len(session_table)} recordings")
problems = session_table[session_table['problem'] != '']
print('Every participant has one session per mapping and per half.' if problems.empty
      else f'*** {problems["id_subject"].nunique()} participant(s) with a problem - see above ***')

# The counterbalancing: how many participants started with each mapping
starters = (session_table[session_table['mapping_order'] == 1]
            .groupby('response_mapping')['id_subject'].apply(sorted))
for mapping, subjects in starters.items():
    print(f"  started {mapping}: {len(subjects)}")

,id_subject,mapping_order,response_mapping,session_half,mapping_first,recorded_at,file,problem
0,E11001,1,upL,1,upL,None,E11001_upL_half1,
1,E11001,2,upR,2,upL,None,E11001_upR_half2,
2,E11002,1,upR,1,upR,None,E11002_upR_half1,
3,E11002,2,upL,2,upR,None,E11002_upL_half2,
4,E11003,1,upL,1,upL,None,E11003_upL_half1,
...,...,...,...,...,...,...,...,...
105,E11058,2,upL,2,upR,None,E11058_upL_half2,
106,E11059,1,upL,1,upL,None,E11059_upL_half1,
107,E11059,2,upR,2,upL,None,E11059_upR_half2,
108,E11060,1,upR,1,upR,None,E11060_upR_half1,


55 participants, 110 recordings
Every participant has one session per mapping and per half.
  started upL: 26
  started upR: 29


## Main processing

One participant at a time, both of their sessions, then one long-format file per participant.

- **Training removed** — the cut is made at the *last* `TRIALID 1` and the trial count after it is
  checked against the expected 100; a count that is not 100 warns.
- **`resolve_session_markers`** refuses a session whose `RESPONSE_MAPPING` or `SESSION_HALF` marker
  disagrees with its filename, rather than picking one.
- **Sounds** — 200 per session, of which ~190 non-empty. Empty (catch) sounds are kept as rows with
  `sound_nonempty = 0` and dropped later by `config.build_filter`.

In [5]:
# ====================================================================
# MAIN PROCESSING
# ====================================================================

for subject, subject_sessions in sessions.items():
    print(f"\n=== {subject} ===")
    processed = [pe11.process_session(s, cfg=cfg, eye=EYE) for s in subject_sessions]
    pe11.generate_long_format_ex11(subject, processed, cfg=cfg)

print('\n' + '=' * 70)
print('COLLECTING ALL INDIVIDUAL FILES')
print('=' * 70)
df_all = pe11.collect_all_long_files_ex11(cfg=cfg)


=== E11001 ===
  E11001_upL_half1  (upL, half 1, session 1 of 2)


  Training removed: experiment starts at the last of 3 'TRIALID 1' markers (line 132343), 100 trials follow


    200 sounds (190 non-empty), 174 keypresses


    200 blinks, 4205 fixations, 4125 saccades (unfiltered)
  E11001_upR_half2  (upR, half 2, session 2 of 2)


  Training removed: experiment starts at the last of 3 'TRIALID 1' markers (line 139557), 100 trials follow


    200 sounds (190 non-empty), 164 keypresses


... (371 outputs left out)

    200 sounds (190 non-empty), 180 keypresses


    436 blinks, 4328 fixations, 4263 saccades (unfiltered)


  Saved data_preprocessed/EX_11/long_format_ex11\E11060_long_ex11.csv (4945 rows, 200 trials, 2 sessions)

COLLECTING ALL INDIVIDUAL FILES


Collected 55 files into data_preprocessed/EX_11\long_format_ex11_all.csv
Total: 249,826 rows, 55 subjects, 110 sessions, 11,000 trials


## Data validation

`trial_number` runs 1..200 across the pair and `id_sacc` / `id_sound` are offset by the first
session's counts, so `groupby(['id_subject', 'trial_number'])` cannot collide two trials from
different sessions. Raw timestamps are **not** made global — the two recordings have separate
tracker clocks — so never compare `sacc_start` across `session_file`s; every analysis quantity is a
latency computed inside one trial.

In [6]:
# ====================================================================
# DATA VALIDATION
# ====================================================================

df_all = pd.read_csv(f"data_preprocessed/{cfg['name']}/{cfg['long_format_file']}", low_memory=False)

print('=== DATA VALIDATION ===')
print(f"Rows                : {len(df_all):,}")
print(f"Participants        : {df_all['id_subject'].nunique()}")
print(f"Sessions            : {df_all['session_file'].nunique()}")
print(f"Trials (per participant): {df_all.groupby('id_subject')['trial_number'].nunique().unique()}")
print(f"\nRows with no saccade : {df_all['id_sacc'].isna().sum()}")

# Uniqueness across the pair: an id in BOTH of a participant's sessions would mean the offsets did
# not apply, and every downstream groupby would silently pool two different trials.
print('\n=== numbering across the session pair ===')
for name, col in (('id_sacc', 'id_sacc'), ('trial_number', 'trial_number'), ('id_sound', 'id_sound')):
    counts = df_all.dropna(subset=[col]).groupby(['id_subject', col])['session_file'].nunique()
    n_shared = int((counts > 1).sum())
    print(f'  {name:<13} values shared by both sessions of one participant: {n_shared}'
          + ('   (must be 0)' if n_shared else '   ok'))

print(f"\nblock_number_global range: {df_all['block_number_global'].min()}"
      f"-{df_all['block_number_global'].max()}")

=== DATA VALIDATION ===
Rows                : 249,826
Participants        : 55
Sessions            : 110
Trials (per participant): [200]

Rows with no saccade : 50

=== numbering across the session pair ===


  id_sacc       values shared by both sessions of one participant: 0   ok
  trial_number  values shared by both sessions of one participant: 0   ok


  id_sound      values shared by both sessions of one participant: 0   ok

block_number_global range: 1-10


### The key mapping actually decoded

The experiment script writes the same facts twice — into the EDF as markers and into its own
behavioural `.csv` — and `check_against_behavioural` compares the two, session by session:

- **sounds** — the `.csv`'s `first_sound_type` / `second_sound_type` against `sound_N_direction`
- **keys** — the `.csv`'s raw `up`/`down` against the key this pipeline says produced its decoded
  side, via `encode_answer_key`

The second is the real test: it goes *side → key*, so a session decoded the wrong way round shows
**0 %** agreement, not 50 %. Both columns must read 1.0 for every session. The function skips a
session with no `.csv`, so the session count is printed too.

In [7]:
# ====================================================================
# RESPONSE DECODING: key -> side
# ====================================================================

decoding = pe11.check_against_behavioural(df_all, cfg=cfg)
checked = decoding[decoding['problem'] == '']
print(f"{len(checked)} of {df_all['session_file'].nunique()} sessions checked against their .csv")
print(f"sound agreement: min {checked['sound_agreement'].min():.3f}   "
      f"key agreement: min {checked['key_agreement'].min():.3f}")

bad = checked[(checked['sound_agreement'] < 1) | (checked['key_agreement'] < 1)]
print('All checked sessions agree with the behavioural .csv exactly.' if bad.empty
      else '*** DO NOT USE THESE DATA UNTIL THIS IS RESOLVED ***')
if not bad.empty:
    display(bad)

# Sessions listed as `not in the long format` belong to participants excluded above.
display(decoding[decoding['problem'] != ''][['session_file', 'problem']])

D:\UJ\projects\audiSacPaper\src\preprocessing_ex11.py:970: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


110 of 110 sessions checked against their .csv
sound agreement: min 1.000   key agreement: min 1.000
All checked sessions agree with the behavioural .csv exactly.


,session_file,problem
28,E11016_upL_half1,*** not in the long format ***
29,E11016_upR_half1,*** not in the long format ***
68,E11037_upL_half1,*** not in the long format ***
99,E11053_upR_half1,*** not in the long format ***
100,E11053_upR_half2,*** not in the long format ***


## Behaviour per participant, and the exclusions applied in the analyses

The preprocessed table is the complete record. The analysis notebooks then apply Experiment 5's
two data-quality exclusions through `src/exclusions.py`, and this cell shows the evidence for both:

- **reversed blocks** — a block whose choice accuracy is below chance (one-sided binomial,
  Benjamini–Hochberg across all blocks) measures the reversed mapping, so it is dropped while the
  participant's other blocks are kept;
- **participants** — E11004 (responded to 28 % of the sounds under `upL`) and E11010 (whole `upR`
  session below chance, i.e. answered with the mapping reversed throughout).

In [8]:
# ====================================================================
# RESPONSE RATE AND ACCURACY PER PARTICIPANT x MAPPING; REVERSED BLOCKS
# ====================================================================

sounds = exclusions.make_sound_table(df_all)
per_session = (sounds.groupby(['id_subject', 'response_mapping'])
               .agg(n_sounds=('acc', 'size'),
                    p_response_given=('response_given', 'mean'),
                    p_correct_choice=('acc', lambda a: a[sounds.loc[a.index, 'response_given'] == 1].mean()))
               .reset_index())
print('Lowest response rates:')
display(per_session.nsmallest(5, 'p_response_given').round(3))
print('Lowest choice accuracies:')
display(per_session.nsmallest(5, 'p_correct_choice').round(3))

screen = exclusions.find_reversed_blocks(df_all, q=exclusions.BLOCK_FDR_Q)
print(f"Reversed blocks (FDR q < {exclusions.BLOCK_FDR_Q}): {int(screen['reversed'].sum())} of {len(screen)}")
display(screen[screen['reversed']].round(3))
print('Next-lowest blocks:')
display(screen[~screen['reversed']].nsmallest(3, 'p_fdr').round(3))

_, _ = exclusions.apply_exp5_exclusions(df_all)

Lowest response rates:


,id_subject,response_mapping,n_sounds,p_response_given,p_correct_choice
6,E11004,upL,190,0.279,0.755
26,E11015,upL,190,0.458,0.931
74,E11042,upL,190,0.521,0.586
27,E11015,upR,190,0.532,0.950
75,E11042,upR,190,0.532,0.673


Lowest choice accuracies:


,id_subject,response_mapping,n_sounds,p_response_given,p_correct_choice
19,E11010,upR,190,0.737,0.379
59,E11033,upR,190,0.984,0.492
81,E11045,upR,190,0.816,0.516
44,E11025,upL,190,0.900,0.561
80,E11045,upL,190,0.842,0.562


Reversed blocks (FDR q < 0.05): 3 of 550


,id_subject,session_file,response_mapping,block_number,block_number_global,n_responded,n_correct,p_correct_choice,p_below_chance,p_fdr,reversed
291,E11033,E11033_upR_half1,upR,2,2,38,8,0.211,0.0,0.043,True
292,E11033,E11033_upR_half1,upR,3,3,39,2,0.051,0.0,0.000,True
428,E11047,E11047_upR_half2,upR,4,9,37,3,0.081,0.0,0.000,True


Next-lowest blocks:


,id_subject,session_file,response_mapping,block_number,block_number_global,n_responded,n_correct,p_correct_choice,p_below_chance,p_fdr,reversed
96,E11010,E11010_upR_half2,upR,2,7,29,7,0.241,0.004,0.559,False
0,E11001,E11001_upL_half1,upL,1,1,33,25,0.758,0.999,1.000,False
1,E11001,E11001_upL_half1,upL,2,2,35,27,0.771,1.000,1.000,False


Exp 5: reversed blocks dropped: 3 of 550 (E11033 upR block 2, E11033 upR block 3, E11047 upR block 4)
Exp 5: participants excluded: E11004, E11010 -> 53 analysed


## Output

```
data_preprocessed/EX_11/
  long_format_ex11/<ID>_long_ex11.csv    one file per participant, BOTH mappings
  long_format_ex11_all.csv               the aggregated table every analysis loads
```

`config.load_experiment_data('EX_11')` reads it (the file name comes from
`EXPERIMENT_CONFIGS['EX_11']['data_file']`); the analysis notebooks then call
`exclusions.apply_exp5_exclusions`.

**If you re-run this notebook after dropping a participant**, delete their `_long_ex11.csv` as
well: the aggregation globs whatever is on disk, whether or not the `.asc` still exists.